# Presentacion — Asistente Fiscal GestorIA
**Tiempo estimado: 7-8 minutos**

> Cada celda markdown = bloque de discurso. Las celdas de codigo = demos en vivo.
> Ejecuta primero todas las celdas del notebook principal (agente_fiscal.ipynb) para que el agente este listo.


## 1. El problema (30 seg)

Una gestoria maneja decenas de clientes — cada uno con sus propias obligaciones fiscales.

El problema no es saber que hay que presentar, sino **cuando hay que empezar a prepararlo**:

- Modelo 200 (Impuesto Sociedades): 30 dias de preparacion
- Modelo 303 (IVA trimestral): 10 dias
- Si el gestor llega tarde, el cliente tiene una sancion.

**Solucion:** un agente de IA que conoce el calendario fiscal, los manuales de la AEAT y el perfil de cada cliente.


## 2. Stack tecnologico (30 seg)

| Componente | Tecnologia |
|---|---|
| LLM | Google Gemini — fallback automatico por modelo y clave |
| Base de conocimiento | ChromaDB — 2.158 chunks de 9 fuentes |
| Embeddings | sentence-transformers multilingue (local, sin coste de API) |
| Agente | LangGraph — grafo con routing condicional |
| Interfaz | Streamlit Cloud (en produccion) |

**Punto clave:** si gemini-3-flash-preview agota su cuota diaria, cae automaticamente a gemini-2.5-flash, etc.
Sin interrupciones, sin errores visibles al usuario.


## 3. Arquitectura del agente (1 min)

```
START
  podar_historial     limpia mensajes antiguos (evita desbordar el contexto)
  detectar_perfil     autonomo o sociedad? (se detecta una vez, persiste toda la sesion)
  clasificar_consulta plazo / casillas / general?
  recuperar_[tipo]    RAG especializado segun tipo de pregunta
  generar_respuesta   Gemini + contexto RAG + historial
```

- Preguntas de plazo priorizan el CSV del calendario
- Preguntas de casillas priorizan los PDFs de manuales
- Moderacion en cascada: regex -> ML TF-IDF -> LLM ligero. El 95% sin tocar el LLM principal.


## 4. Base de conocimiento (30 seg)

**2.158 chunks** indexados en ChromaDB:

- 7 PDFs de manuales AEAT (IVA, Renta, Sociedades, Actividades Economicas)
- Calendario fiscal 2026 con plazos, domiciliaciones y dias de preparacion recomendados
- Mapa de obligaciones por perfil de contribuyente

**SemanticChunker (percentil 95):** divide respetando fronteras conceptuales.
Cada chunk lleva metadatos: perfil, modelo, tipo, fuente — para filtrar en la recuperacion.


## 5. DEMO A — Autonomo, primer trimestre (2 min)

**Decir antes de ejecutar:**

Simulamos una consulta real de un autonomo en estimacion directa.
Observad tres cosas:
1. El agente detecta el perfil en la primera pregunta y lo recuerda en las siguientes
2. Muestra siempre la fecha de domiciliacion ademas del plazo limite
3. La cuarta pregunta no menciona el modelo 130 — el agente lo recuerda del contexto anterior


In [ ]:
import uuid
from langchain_core.messages import HumanMessage

def preguntar(pregunta, state, config):
    state["messages"] = state.get("messages", []) + [HumanMessage(content=pregunta)]
    result = agente.invoke(state, config=config)
    return result["messages"][-1].content, result

thread_a = str(uuid.uuid4())
config_a = {"configurable": {"thread_id": thread_a}}
state_a  = {"messages": [], "perfil": "", "contexto_rag": "", "tipo_consulta": ""}

preguntas_a = [
    "Soy autonomo en estimacion directa. Que declaraciones tengo en el primer trimestre de 2026?",
    "Cual es la fecha limite del modelo 130 y cuando deberia empezar a prepararlo?",
    "Como calculo el importe a ingresar en el modelo 130?",
    "Y si tengo retenciones de clientes, como las resto?",
]

for i, p in enumerate(preguntas_a, 1):
    sep = "-" * 50
    print(f"\n{sep}")
    print(f"Pregunta {i}: {p}")
    print(sep)
    respuesta, state_a = preguntar(p, state_a, config_a)
    print(f"Agente:\n{respuesta}\n")


## 6. DEMO B — Sociedad, cierre de ejercicio (2 min)

**Decir antes de ejecutar:**

Ahora una S.L. a finales de ano — el momento mas critico del calendario fiscal.
Fijate en:
1. Lista todas las obligaciones del 4T incluyendo el cierre de enero
2. Calcula 30 dias de antelacion para el modelo 200 (el mas complejo)
3. La cuarta pregunta no repite que es sociedad — el agente lo recuerda


In [ ]:
thread_b = str(uuid.uuid4())
config_b = {"configurable": {"thread_id": thread_b}}
state_b  = {"messages": [], "perfil": "", "contexto_rag": "", "tipo_consulta": ""}

preguntas_b = [
    "Somos una S.L. Que obligaciones tenemos en el cuarto trimestre de 2026 y principios de 2027?",
    "Cuando hay que presentar el Impuesto de Sociedades (modelo 200) y con cuanta antelacion?",
    "Hay que hacer pago fraccionado del IS en octubre? Que es el modelo 202?",
    "Como se calcula la base del pago fraccionado del modelo 202?",
]

for i, p in enumerate(preguntas_b, 1):
    sep = "-" * 50
    print(f"\n{sep}")
    print(f"Pregunta {i}: {p}")
    print(sep)
    respuesta, state_b = preguntar(p, state_b, config_b)
    print(f"Agente:\n{respuesta}\n")


## 7. Streamlit Cloud (30 seg)

Abrir en vivo: **https://agentexpertofiscalidad.streamlit.app/**

Mostrar:
- Sidebar con selector de perfil (Autonomo / Sociedad)
- Tab "Calendario Fiscal" — semaforo de urgencia por dias restantes
- Toggle "Mostrar evaluacion" — LLM-as-Judge puntua precision, claridad y completitud


## 8. Decisiones tecnicas destacables (1 min)

Elegir 2-3 segun el tiempo disponible:

**a) SemanticChunker en lugar de tamano fijo**
Un manual fiscal tiene apartados de longitud muy variable.
Cortar a 500 tokens fijos parte explicaciones de casillas por la mitad e inutiliza el chunk.
El SemanticChunker detecta donde termina un concepto y empieza otro.

**b) Moderacion en cascada**
Tres capas: regex (gratis) -> TF-IDF ML (microsegundos) -> LLM ligero (solo ambiguos).
El 95% de preguntas se clasifican sin tocar el LLM principal — ahorra cuota y reduce latencia.

**c) Fallback modelo x clave**
Con cuota gratuita de Gemini, un solo modelo se agota rapido.
La matriz prueba todas las claves del modelo antes de bajar al siguiente. Transparente al usuario.

**d) LLM-as-Judge integrado**
Segundo LLM evalua cada respuesta: precision tecnica, claridad y completitud.
Permite detectar respuestas deficientes sin revision manual.


## 9. Cierre (30 seg)

- 9 modelos fiscales cubiertos para autonomos y sociedades
- Responde en el idioma del usuario (castellano, catalan, euskera, ingles, etc.)
- En produccion en Streamlit Cloud, codigo en GitHub

**Limitacion honesta (si preguntan):**
Las instrucciones casilla a casilla del modelo 130 no estan en los PDFs indexados.
El manual de Renta cubre el modelo 100 anual, no el 130 trimestral.
Siguiente paso: anadir las instrucciones especificas del 130 desde la sede AEAT.

---

**Preguntas frecuentes:**

*Por que LangGraph y no LangChain directamente?*
El grafo permite depurar nodo a nodo, ampliar sin reescribir y la memoria con MemorySaver es nativa.

*Por que ChromaDB y no FAISS?*
ChromaDB persiste en disco y filtra por metadatos (perfil, modelo). FAISS requiere mas codigo para eso.

*Como garantizas que no alucina fechas?*
El system prompt prohibe inventar datos fuera del RAG. El calendario es la fuente de maxima autoridad. Temperature=0.
